# LookOut — Thief/Robbery Model Training (Colab GPU)

Trains the 4-class thief detector (`gun`, `knife`, `robbery activity`, `stealing`) on a free Colab GPU in ~15–25 min.

**Before running:**
1. `Runtime → Change runtime type → T4 GPU`
2. Upload `thief_dataset_cleaned.zip` (from your Downloads folder) to your **Google Drive** root — the direct-upload widget is too slow for ~300 MB.

**After it finishes:** the last cell downloads `best.pt`. Rename it to `thief.pt` and copy it to `lookout_backend/core/vision/thief.pt` on your laptop.

In [ ]:
# 1. Check the GPU is on (should print a Tesla T4 table; if it errors, fix the runtime type first)
!nvidia-smi

In [ ]:
# 2. Mount Google Drive and unzip the dataset to /content/dataset
from google.colab import drive
drive.mount('/content/drive')

!unzip -q "/content/drive/MyDrive/thief_dataset_cleaned.zip" -d /content/dataset
!ls /content/dataset

In [ ]:
# 3. Install ultralytics
%pip install -q ultralytics

In [ ]:
# 4. Train — GPU lets us afford better settings than the laptop:
#    imgsz 640 (vs 416) and 50 epochs (vs 20) for noticeably higher accuracy.
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
model.train(
    data='/content/dataset/data_colab.yaml',
    epochs=50,
    imgsz=640,
    batch=32,
    device=0,
    patience=15,
    project='/content/runs',
    name='thief',
    exist_ok=True,
)

In [ ]:
# 5. Evaluate on the held-out test split (per-class mAP — expect 'stealing' to be weakest)
metrics = YOLO('/content/runs/thief/weights/best.pt').val(
    data='/content/dataset/data_colab.yaml', split='test'
)

In [ ]:
# 6. Download the trained weights (also copies a backup to your Drive).
#    On your laptop: rename to thief.pt -> lookout_backend/core/vision/thief.pt
!cp /content/runs/thief/weights/best.pt /content/drive/MyDrive/thief_best.pt
from google.colab import files
files.download('/content/runs/thief/weights/best.pt')